In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import make_scorer, recall_score
from pathlib import Path

LOCAL_PATH = Path("../data/parkinsons.data")
URL = "https://raw.githubusercontent.com/youelfissha23-boop/parkinsons-voice-screening/main/data/parkinsons.data"

df = pd.read_csv(LOCAL_PATH if LOCAL_PATH.exists() else URL)
print(df.shape)
display(df.head())
df.info()

(195, 24)


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              195 non-null    object 
 1   MDVP:Fo(Hz)       195 non-null    float64
 2   MDVP:Fhi(Hz)      195 non-null    float64
 3   MDVP:Flo(Hz)      195 non-null    float64
 4   MDVP:Jitter(%)    195 non-null    float64
 5   MDVP:Jitter(Abs)  195 non-null    float64
 6   MDVP:RAP          195 non-null    float64
 7   MDVP:PPQ          195 non-null    float64
 8   Jitter:DDP        195 non-null    float64
 9   MDVP:Shimmer      195 non-null    float64
 10  MDVP:Shimmer(dB)  195 non-null    float64
 11  Shimmer:APQ3      195 non-null    float64
 12  Shimmer:APQ5      195 non-null    float64
 13  MDVP:APQ          195 non-null    float64
 14  Shimmer:DDA       195 non-null    float64
 15  NHR               195 non-null    float64
 16  HNR               195 non-null    float64
 1

In [2]:
print(df["status"].value_counts())
print(df["status"].value_counts(normalize=True))

status
1    147
0     48
Name: count, dtype: int64
status
1    0.753846
0    0.246154
Name: proportion, dtype: float64


In [3]:
df["subject"] = df["name"].str.extract(r"(S\d+)")
df["subject"].nunique()
df.groupby("subject")["status"].nunique().max()

1

In [4]:
y = df["status"]
X = df.drop(columns=["name","subject","status"])
groups = df["subject"]

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
leaky = make_pipeline(StandardScaler(), RandomForestClassifier(random_state=42))
leaky.fit(X_tr, y_tr)
y_pred = leaky.predict(X_te)
print(accuracy_score(y_te, y_pred))

0.9487179487179487


In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [7]:
print("Overlap of people:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

Overlap of people: set()


In [8]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy:", accuracy_score(y_test, dummy.predict(X_test)))

Baseline accuracy: 0.7209302325581395


In [9]:
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("Grouped-split accuracy:", accuracy_score(y_test, preds))
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

Grouped-split accuracy: 0.5813953488372093
[[ 3  9]
 [ 9 22]]
              precision    recall  f1-score   support

           0       0.25      0.25      0.25        12
           1       0.71      0.71      0.71        31

    accuracy                           0.58        43
   macro avg       0.48      0.48      0.48        43
weighted avg       0.58      0.58      0.58        43



In [10]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight="balanced", probability=True)),
    "Random Forest": make_pipeline(StandardScaler(), RandomForestClassifier(class_weight="balanced", random_state=42)),
}
cv = GroupKFold(n_splits=5)
for name, m in models.items():
    r = cross_validate(m, X, y, groups=groups, cv=cv,
                       scoring=["accuracy", "balanced_accuracy", "recall"])
    print(f"{name:20s} acc={r['test_accuracy'].mean():.3f}  "
          f"balanced_acc={r['test_balanced_accuracy'].mean():.3f}  "
          f"recall={r['test_recall'].mean():.3f}")

Logistic Regression  acc=0.709  balanced_acc=0.635  recall=0.770
SVM                  acc=0.650  balanced_acc=0.586  recall=0.717
Random Forest        acc=0.760  balanced_acc=0.615  recall=0.901


In [11]:
demo = pd.DataFrame({"person": list("AAABBBCCCDDD"), "value": range(12)})
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr, te = next(splitter.split(demo, groups=demo["person"]))
print(tr)
print(te)
print(demo.iloc[te])

[ 0  1  2  3  4  5  9 10 11]
[6 7 8]
  person  value
6      C      6
7      C      7
8      C      8


In [12]:
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=0)

for fold, (train_idx, text_idx) in enumerate(cv.split(X, y, groups)):
  test_people = sorted(set(groups.iloc[test_idx]))
  healthy_people = groups.iloc[test_idx][y.iloc[test_idx] == 0].nunique()
  print(f"Fold {fold}: {len(train_idx)} train rows, {len(test_idx)} test rows, "
        f"{len(test_people)} test people, of whom {healthy_people} healthy")

Fold 0: 147 train rows, 43 test rows, 7 test people, of whom 2 healthy
Fold 1: 145 train rows, 43 test rows, 7 test people, of whom 2 healthy
Fold 2: 146 train rows, 43 test rows, 7 test people, of whom 2 healthy
Fold 3: 147 train rows, 43 test rows, 7 test people, of whom 2 healthy


In [13]:
probs = model.predict_proba(X_test)
print(probs.shape)
print(probs[:5])
print(probs[:5].sum(axis=1))

(43, 2)
[[0.06588937 0.93411063]
 [0.11238249 0.88761751]
 [0.1696087  0.8303913 ]
 [0.05392511 0.94607489]
 [0.51310424 0.48689576]]
[1. 1. 1. 1. 1.]


In [14]:
# Initialize an array to hold out-of-fold predictions
oof_proba = np.full(len(X), np.nan)
# Define the cross-validation splitter
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=0)
# Iterate through the folds
for train_idx, test_idx in cv.split(X, y, groups):
    fold_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
    # Split data
    X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
    y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
    # Fit model on training fold
    fold_model.fit(X_train_fold, y_train_fold)
    # Predict on test fold and store in the respective indices
    oof_proba[test_idx] = fold_model.predict_proba(X_test_fold)[:, 1]

oof_pred = (oof_proba >= 0.5).astype(int)
# Evaluation
print("Empty slots left:", np.isnan(oof_proba).sum())
print(oof_proba[:5])
print("Out-of-Fold Accuracy:", accuracy_score(y, oof_pred))

Empty slots left: 0
[0.99991586 0.99999596 0.99997379 0.99999343 0.99996683]
Out-of-Fold Accuracy: 0.7487179487179487


In [15]:
print(confusion_matrix(oof_pred, y))

[[ 26  27]
 [ 22 120]]


In [16]:
#step D
tn, fp, fn, tp = confusion_matrix(y, oof_pred).ravel()

recall = tp/ (tp + fn)
specificity = tn/ (tn + fp)
print(f"recall: {recall}, specificity: {specificity}")

recall: 0.8163265306122449, specificity: 0.5416666666666666


In [17]:
#step E
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
  pred = (oof_proba >= t).astype(int)
  tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
  recall = tp/ (tp + fn)
  specificity = tn/ (tn + fp)
  print(f"recall: {recall}")
  print(f"specificity: {specificity}")
  print("flagged:", pred.sum())

recall: 0.8571428571428571
specificity: 0.4791666666666667
flagged: 151
recall: 0.8299319727891157
specificity: 0.5
flagged: 146
recall: 0.8163265306122449
specificity: 0.5416666666666666
flagged: 142
recall: 0.782312925170068
specificity: 0.5416666666666666
flagged: 137
recall: 0.7346938775510204
specificity: 0.5625
flagged: 129
recall: 0.6666666666666666
specificity: 0.5833333333333334
flagged: 118
